# Single-pass JEPA on Qwen3-1.7B with LoRA: NOUL + choice decisions

**The decision** is read exactly as in Signpost-D: the prompt (state, question with labelled options, the reply line) through the model, and the first answer token's logits over the question's own labels (`yes`/`no`, or `A`, `B`, …), softmaxed, and trained with KL to the gold distribution.

**The JEPA loss comes from the same forward pass.** During training the sequence is the prompt **followed by the gold answer token** (the gold's most likely label). The model is causal, so:
- **Context:** layer `layer`'s hidden state at the prompt's last token, which has not seen the answer → a predictor → ẑ.
- **Target:** layer `layer`'s hidden state at the answer token, which has read the prompt and the answer: the model's representation of the decided question. **Stop-gradient.**
- **Loss:** KL + `jepa_weight` × (1 − cos(ẑ, target)), both standardised per dimension (frozen-base statistics), since a few of Qwen's dimensions are huge.

**`target_mode`:** `same` (one pass; the target is the model itself) or `ema` (two passes; the target comes from an **EMA copy** of the LoRA weights, decay `ema_decay`, run without gradient: the classic context encoder / EMA target encoder pair). In `same` mode the context side and the target side are the same LoRA-tuned model, so both are fine-tuned: the target improves as the model does, without receiving the JEPA gradient itself. **At inference** only the decision read-out runs (no answer token, no predictor).

**Data:** all 2,575 training records of `signpost-d-decisions` (NOUL and choice questions), the 200 dev records for validation, and the Typed Decisions test split's NOUL and choice questions for the benchmark. Signpost-D-4B's public predictions on the same questions are the reference.

## Settings

In [ ]:
import os


def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        kind = type(default[0]) if default else str
        return [kind(v.strip()) for v in value.split(",") if v.strip()]
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    return type(default)(value)


CONFIG = {
    "seed": setting("seed", 42),
    "model_name": setting("model_name", "Qwen/Qwen3-1.7B"),
    "layer": setting("layer", 20),
    "types": setting("types", ["noul", "choice"]),
    "data_base": setting("data_base", "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                                      "decider-jepa/data/signpost_d_decisions/"),
    "max_tokens": setting("max_tokens", 2048),

    # LoRA (the Signpost-D settings)
    "lora_r": setting("lora_r", 16),
    "lora_alpha": setting("lora_alpha", 32),
    "lora_dropout": setting("lora_dropout", 0.05),
    "lora_targets": setting("lora_targets", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),

    # JEPA
    "jepa_weight": setting("jepa_weight", 1.0),
    "choice_readout": setting("choice_readout", "letter"),   # "letter": the first answer token's logits over A, B, ...
                                                              # "option_line": a scorer on each option line's last token
    "target_mode": setting("target_mode", "same"),      # "same": one pass, the model itself; "ema": a second pass
                                                         # through an EMA copy of the LoRA weights (no gradient)
    "ema_decay": setting("ema_decay", 0.996),
    "predictor_hidden": setting("predictor_hidden", 2048),

    # training
    "epochs": setting("epochs", 2),
    "batch_size": setting("batch_size", 8),
    "eval_batch": setting("eval_batch", 16),
    "lora_lr": setting("lora_lr", 1e-4),
    "predictor_lr": setting("predictor_lr", 3e-4),
    "weight_decay": setting("weight_decay", 0.01),
    "warmup_fraction": setting("warmup_fraction", 0.15),
    "grad_clip": setting("grad_clip", 1.0),
    "checkpoint_every": setting("checkpoint_every", 250),
    "speed_check_steps": setting("speed_check_steps", 50),

    "td_dataset": setting("td_dataset", "LocalLLaMA/typed-decisions"),
    "td_file": setting("td_file", "all/test-00000-of-00001.parquet"),
    "reference_model": setting("reference_model", "skamalj/signpost-d-4b"),
    "reference_file": setting("reference_file", "typed_decisions_test_predictions.json"),
    "out_dir": setting("out_dir", "jepa_single"),
}

for k, v in CONFIG.items():
    print(f"{k:22s}: {v}")

## Data: questions rendered as in Signpost-D

NOUL: options `yes` (the criteria's `true` text) and `no` (`false`). Choice: options labelled `A`, `B`, … with the option name and description; in training the options are shuffled every epoch (so the model cannot learn positions). The gold distribution follows the options.

In [ ]:
import json
import random
import urllib.request

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = torch.device("cuda")

LABELS = {"noul": ["yes", "no"], "choice": [chr(65 + i) for i in range(26)]}
REPLY = {"noul": "Answer with yes or no only.", "choice": "Answer with the letter of one option only."}


def parsed(x):
    return json.loads(x) if isinstance(x, str) else x


def items_of(record, source):
    state = parsed(record["state"])
    state = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
    gold_all = parsed(record["gold"])
    out = []
    for qid, q in parsed(record["questions"]).items():
        kind = str(q.get("type", "")).lower()
        if kind not in CONFIG["types"]:
            continue
        criteria = q.get("criteria")
        if kind == "noul":
            criteria = criteria or {}
            options = {"yes": criteria.get("true"), "no": criteria.get("false")}
            keys = ["true", "false"]
        else:
            options = {str(k): (None if v is None else str(v)) for k, v in criteria.items()}
            keys = list(criteria)
        if len(options) > 26:
            continue
        gold = gold_all[qid]["probabilities"]
        out.append({"type": kind, "id": record["id"], "qid": qid, "state": state, "source": source,
                    "workflow": record.get("workflow", ""),
                    "question": "" if q.get("instructions") is None else str(q["instructions"]),
                    "options": options, "soft": {name: float(gold[key]) for name, key in zip(options, keys)}})
    return out


def our_split(name):
    text = urllib.request.urlopen(CONFIG["data_base"] + f"{name}.jsonl").read().decode("utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


train_items = [it for r in our_split("train") for it in items_of(r, "train")]
dev_items = [it for r in our_split("dev") for it in items_of(r, "dev")]
td_frame = pd.read_parquet(hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset"))
td_items = [it for r in td_frame.to_dict("records") for it in items_of(r, "td")]


def counts(items):
    return dict(pd.Series([it["type"] for it in items]).value_counts())


print("train:", len(train_items), counts(train_items))
print("dev  :", len(dev_items), counts(dev_items))
print("td   :", len(td_items), counts(td_items))

## Prompt, label tokens, model

In [ ]:
import math
import time

import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], dtype=torch.bfloat16).to(device)
HIDDEN, LAYER = model.config.hidden_size, CONFIG["layer"]


def question_lines(item):
    lines = [f"Question: {item['question']}"] if item["question"] else []
    option_at = []
    for label, (name, description) in zip(LABELS[item["type"]], item["options"].items()):
        option_at.append(len(lines))
        if item["type"] == "noul":
            lines.append(f"{label}: {description}" if description else label)
        else:
            lines.append(f"{label}: {name} - {description}" if description else f"{label}: {name}")
    lines.append(REPLY[item["type"]])
    return lines, option_at


def prompt_text(item):
    lines, _ = question_lines(item)
    content = f"{item['state']}\n\n" + "\n".join(lines)
    return tokenizer.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                         tokenize=False, enable_thinking=False)


def prompt_ids_and_option_ends(item):
    """-> (token ids of the prompt, index of the last token of each option line)."""
    text = prompt_text(item)
    lines, option_at = question_lines(item)
    block = "\n".join(lines)
    start = text.rindex(block)
    line_start = [start + sum(len(l) + 1 for l in lines[:i]) for i in range(len(lines))]
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ends = []
    for i in option_at:
        last_char = line_start[i] + len(lines[i]) - 1
        offsets = enc["offset_mapping"]
        inside = [t for t, (a, b) in enumerate(offsets) if a <= last_char < b]
        ends.append(inside[-1] if inside else max(t for t, (a, b) in enumerate(offsets) if a < b <= last_char + 1))
    return enc["input_ids"], ends


# each label must be exactly one token right after the generation prompt
probe = prompt_text({"type": "noul", "question": "x", "options": {"yes": None, "no": None}, "state": "{}"})
probe_ids = tokenizer(probe, add_special_tokens=False)["input_ids"]
LABEL_IDS = {}
for kind, labels in LABELS.items():
    ids = []
    for label in labels:
        extended = tokenizer(probe + label, add_special_tokens=False)["input_ids"]
        assert extended[:-1] == probe_ids and extended[-1] not in ids, f"label {label!r} is not one token"
        ids.append(extended[-1])
    LABEL_IDS[kind] = ids
print({k: v[:5] for k, v in LABEL_IDS.items()})

lengths = [len(tokenizer(prompt_text(it), add_special_tokens=False)["input_ids"]) for it in train_items[:500]]
print(f"prompt tokens (500 training items): median {int(np.median(lengths))}, max {max(lengths)}")

## Standardisation statistics (frozen base, before LoRA)

Per-dimension mean and std of layer `layer` at the prompt's last token and at the answer token, over 512 training items. The JEPA cosine is computed in this standardised space.

In [ ]:
def batch_ids(items, with_answer):
    """Right-padded ids; returns (input_ids, attention_mask, prompt_last, answer_pos)."""
    seqs, last, ans, option_ends = [], [], [], []
    for it in items:
        ids, ends = prompt_ids_and_option_ends(it)
        option_ends.append(ends)
        assert len(ids) < CONFIG["max_tokens"], len(ids)
        last.append(len(ids) - 1)
        if with_answer:
            names = list(it["options"])
            gold_index = max(range(len(names)), key=lambda j: it["soft"][names[j]])
            ids = ids + [LABEL_IDS[it["type"]][gold_index]]
        ans.append(len(ids) - 1)
        seqs.append(ids)
    n = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), n), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), n, dtype=torch.long)
    for r, s in enumerate(seqs):
        input_ids[r, :len(s)] = torch.tensor(s)
        mask[r, :len(s)] = 1
    return (input_ids.to(device), mask.to(device), torch.tensor(last, device=device), torch.tensor(ans, device=device),
            option_ends)


with torch.no_grad():
    model.eval()
    sample = random.Random(0).sample(train_items, 512)
    rows = []
    for s in range(0, len(sample), 16):
        ids, mask, last, ans, _ = batch_ids(sample[s:s + 16], with_answer=True)
        hs = model.model(input_ids=ids, attention_mask=mask, output_hidden_states=True).hidden_states[LAYER]
        r = torch.arange(len(ids), device=device)
        rows += [hs[r, last].float(), hs[r, ans].float()]
    rows = torch.cat(rows)
    MEAN, STD = rows.mean(0), rows.std(0).clamp_min(1e-6)
print(f"standardisation from {len(rows)} vectors; largest raw std {STD.max().item():.1f}, median {STD.median().item():.3f}")

## LoRA, the predictor, the losses

In [ ]:
import copy

from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

model = get_peft_model(model, LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
                                         lora_dropout=CONFIG["lora_dropout"], target_modules=CONFIG["lora_targets"],
                                         task_type="CAUSAL_LM"))
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()
DECODER = model.base_model.model.model                 # the decoder stack (with LoRA)
LM_HEAD = model.get_output_embeddings().weight

# target_mode "ema": the target encoder is an EMA copy of the LoRA weights (the base weights are shared, frozen)
LORA_PARAMS = [p for p in model.parameters() if p.requires_grad]
EMA = [p.detach().clone() for p in LORA_PARAMS] if CONFIG["target_mode"] == "ema" else None


@torch.no_grad()
def ema_update():
    for e, p in zip(EMA, LORA_PARAMS):
        e.mul_(CONFIG["ema_decay"]).add_(p.detach(), alpha=1 - CONFIG["ema_decay"])


@torch.no_grad()
def ema_hidden(ids, mask):
    """Layer-`layer` states from the EMA target encoder: swap the EMA LoRA weights in, one pass, swap back."""
    backup = [p.detach().clone() for p in LORA_PARAMS]
    for p, e in zip(LORA_PARAMS, EMA):
        p.copy_(e)
    was_training = model.training
    model.eval()
    hs = DECODER(input_ids=ids, attention_mask=mask, output_hidden_states=True).hidden_states[LAYER]
    model.train(was_training)
    for p, b in zip(LORA_PARAMS, backup):
        p.copy_(b)
    return hs


class Predictor(nn.Module):
    def __init__(self):
        super().__init__()
        h = CONFIG["predictor_hidden"]
        self.net = nn.Sequential(nn.LayerNorm(HIDDEN), nn.Linear(HIDDEN, h), nn.GELU(), nn.Linear(h, HIDDEN))

    def forward(self, x):
        return self.net(x)


predictor = Predictor().to(device)
option_scorer = nn.Sequential(nn.LayerNorm(HIDDEN), nn.Linear(HIDDEN, 1)).to(device)   # choice_readout = option_line


def decision_logits(final, last, option_ends, items):
    """Per item: the label logits at the prompt's last token (yes/no, or letters), or for choice questions with
    choice_readout = option_line, the scorer on each option line's last token. Padded with -1e4."""
    k = max(len(it["options"]) for it in items)
    out = torch.full((len(items), k), -1e4, device=device)     # padded option slots
    for r, it in enumerate(items):
        n = len(it["options"])
        if it["type"] == "choice" and CONFIG["choice_readout"] == "option_line":
            ends = torch.tensor(option_ends[r], device=device)
            out[r, :n] = option_scorer(final[r, ends].float()).squeeze(-1)
        else:
            rows = torch.tensor(LABEL_IDS[it["type"]][:n], device=device)
            out[r, :n] = final[r, last[r]].float() @ LM_HEAD[rows].float().T
    return out


def gold_matrix(items, k):
    g = torch.zeros(len(items), k, device=device)
    for r, it in enumerate(items):
        g[r, :len(it["options"])] = torch.tensor([it["soft"][n] for n in it["options"]], device=device)
    return g / g.sum(1, keepdim=True)


def kl(logits, gold):
    logp = F.log_softmax(logits, dim=1)
    return torch.where(gold > 0, gold * (gold.clamp_min(1e-12).log() - logp), torch.zeros_like(gold)).sum(1)


def forward(items, train):
    """train: prompt + gold answer token (JEPA); eval: prompt only."""
    ids, mask, last, ans, option_ends = batch_ids(items, with_answer=train)
    # the EMA target pass runs first: swapping weights after the online pass would modify tensors autograd needs
    ema_target_hs = ema_hidden(ids, mask) if train and CONFIG["target_mode"] == "ema" else None
    out = DECODER(input_ids=ids, attention_mask=mask, output_hidden_states=True)
    r = torch.arange(len(items), device=device)
    logits = decision_logits(out.last_hidden_state, last, option_ends, items)
    if not train:
        return logits, None, None
    hs = out.hidden_states[LAYER]
    z = predictor((hs[r, last].float() - MEAN) / STD)
    target_hs = ema_target_hs if ema_target_hs is not None else hs
    target = ((target_hs[r, ans].float() - MEAN) / STD).detach()
    return logits, z, target

## Measures

In [ ]:
def summary(rows):
    acc, kls, brier = [], [], []
    for it, p in rows:
        g = it["soft"]
        acc.append(max(p, key=p.get) == max(g, key=g.get))
        kls.append(sum(gv * math.log(gv / max(p[n], 1e-9)) for n, gv in g.items() if gv > 0))
        brier.append(sum((p[n] - g[n]) ** 2 for n in g))
    return {"n": len(rows), "accuracy": round(float(np.mean(acc)), 4), "kl": round(float(np.mean(kls)), 4),
            "brier": round(float(np.mean(brier)), 4)}


def report(rows):
    out = {"all": summary(rows)}
    for t in CONFIG["types"]:
        sub = [(it, p) for it, p in rows if it["type"] == t]
        if sub:
            out[t] = summary(sub)
    return out


@torch.no_grad()
def predict(items):
    model.eval()
    order = sorted(range(len(items)), key=lambda i: len(items[i]["state"]))
    rows = []
    for s in range(0, len(order), CONFIG["eval_batch"]):
        chunk = [items[i] for i in order[s:s + CONFIG["eval_batch"]]]
        logits, _, _ = forward(chunk, train=False)
        probs = F.softmax(logits, dim=1).cpu()
        for it, p in zip(chunk, probs):
            rows.append((it, {n: float(p[j]) for j, n in enumerate(it["options"])}))
    return rows


uniform_rows = [(it, {n: 1 / len(it["options"]) for n in it["options"]}) for it in dev_items]
print("dev, uniform:", report(uniform_rows))

## Training (2 epochs, checkpoints with resume)

Every `log` steps: KL, the JEPA cosine, and the **spread of the targets** (mean per-dimension std across the batch, in the standardised space); a spread falling towards 0 would mean the targets are collapsing.

In [ ]:
from pathlib import Path

OUT_DIR = Path(CONFIG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT, BEST = OUT_DIR / "latest.pt", OUT_DIR / "best.pt"

param_groups = [{"params": [p for p in model.parameters() if p.requires_grad], "lr": CONFIG["lora_lr"]},
                {"params": list(predictor.parameters()) + list(option_scorer.parameters()), "lr": CONFIG["predictor_lr"]}]
opt = torch.optim.AdamW(param_groups, weight_decay=CONFIG["weight_decay"])
steps_per_epoch = len(train_items) // CONFIG["batch_size"]
total_steps = CONFIG["epochs"] * steps_per_epoch
warmup = max(1, int(CONFIG["warmup_fraction"] * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warmup if s < warmup else max(0.0, (total_steps - s) / max(1, total_steps - warmup)))


def epoch_order(epoch):
    """The training items, shuffled per epoch, with choice options shuffled per epoch too."""
    rng = random.Random(f"{CONFIG['seed']}-{epoch}")
    items = []
    for it in train_items:
        it = dict(it)
        if it["type"] == "choice":
            names = list(it["options"])
            rng.shuffle(names)
            it["options"] = {n: it["options"][n] for n in names}
        items.append(it)
    rng.shuffle(items)
    return items


def trained():
    return {"lora": {k: v.detach().cpu() for k, v in get_peft_model_state_dict(model).items()},
            "predictor": {k: v.detach().cpu() for k, v in predictor.state_dict().items()},
            "option_scorer": {k: v.detach().cpu() for k, v in option_scorer.state_dict().items()},
            "ema": [e.cpu() for e in EMA] if EMA is not None else None}


def save(path, obj):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


state = {"step": 0, "best_kl": float("inf"), "best_epoch": 0, "history": []}
if CKPT.exists():
    saved = torch.load(CKPT, map_location="cpu", weights_only=False)
    set_peft_model_state_dict(model, saved["trained"]["lora"]); predictor.load_state_dict(saved["trained"]["predictor"])
    option_scorer.load_state_dict(saved["trained"]["option_scorer"])
    if EMA is not None and saved["trained"].get("ema"):
        for e, v in zip(EMA, saved["trained"]["ema"]):
            e.copy_(v.to(e.device))
    opt.load_state_dict(saved["opt"]); sched.load_state_dict(saved["sched"]); state = saved["state"]
    print(f"RESUMED at step {state['step']}")

RATES = {"L4": 0.80, "A10": 1.10, "L40S": 1.95, "A100": 2.50, "H100": 3.95}
gpu = torch.cuda.get_device_name(0)
rate = next((v for k, v in RATES.items() if k in gpu), None)
print(f"{len(train_items)} items, {steps_per_epoch} steps per epoch, {total_steps} steps | {gpu}")

t_start, window = time.perf_counter(), []
while state["step"] < total_steps:
    epoch = state["step"] // steps_per_epoch + 1
    order = epoch_order(epoch)
    model.train(); predictor.train()
    for s in range(state["step"] % steps_per_epoch, steps_per_epoch):
        items = order[s * CONFIG["batch_size"]:(s + 1) * CONFIG["batch_size"]]
        logits, z, target = forward(items, train=True)
        loss_kl = kl(logits, gold_matrix(items, logits.shape[1])).mean()
        cos = F.cosine_similarity(z, target, dim=1)
        loss = loss_kl + CONFIG["jepa_weight"] * (1 - cos).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for g in param_groups for p in g["params"]], CONFIG["grad_clip"])
        opt.step(); sched.step()
        if EMA is not None:
            ema_update()
        state["step"] += 1
        window.append((loss_kl.item(), cos.mean().item(), target.std(0).mean().item()))
        if state["step"] == CONFIG["speed_check_steps"]:
            per = (time.perf_counter() - t_start) / CONFIG["speed_check_steps"]
            left = per * (total_steps - state["step"]) / 3600
            print(f"SPEED CHECK on {gpu}: {per:.2f} s/step -> {left * 60:.0f} min of training left"
                  + (f", about ${left * rate:.2f} at ${rate}/h" if rate else ""))
        if state["step"] % 100 == 0:
            w = np.array(window[-100:])
            print(f"step {state['step']}/{total_steps} | KL {w[:, 0].mean():.4f} | JEPA cos {w[:, 1].mean():.4f} | "
                  f"target spread {w[:, 2].mean():.3f}")
        if state["step"] % CONFIG["checkpoint_every"] == 0:
            save(CKPT, {"trained": trained(), "opt": opt.state_dict(), "sched": sched.state_dict(), "state": state})
    dev = report(predict(dev_items))
    state["history"].append({"epoch": epoch, "dev": dev})
    print(f"EPOCH {epoch} dev | all {dev['all']} | " + " | ".join(f"{t} {dev[t]}" for t in CONFIG["types"] if t in dev))
    if dev["all"]["kl"] < state["best_kl"]:
        state["best_kl"], state["best_epoch"] = dev["all"]["kl"], epoch
        save(BEST, trained())
    save(CKPT, {"trained": trained(), "opt": opt.state_dict(), "sched": sched.state_dict(), "state": state})

print(f"best epoch {state['best_epoch']} (dev KL {state['best_kl']:.4f})")

## Benchmark: Typed Decisions test, NOUL and choice

In [ ]:
best = torch.load(BEST, map_location="cpu", weights_only=False)
set_peft_model_state_dict(model, best["lora"])
option_scorer.load_state_dict(best["option_scorer"])

dev_result = report(predict(dev_items))
td_rows = predict(td_items)
td_result = report(td_rows)

# reference: Signpost-D-4B's public predictions on the same questions
reference = {}
for p in json.load(open(hf_hub_download(CONFIG["reference_model"], CONFIG["reference_file"]), encoding="utf-8")):
    if p["type"] in CONFIG["types"]:
        reference[(td_frame.iloc[p["case"]]["id"], p["question"])] = p["probabilities"]
ref_rows = []
for it in td_items:
    probs = reference.get((it["id"], it["qid"]))
    if probs is not None:
        ref_rows.append((it, {"yes": probs["yes"], "no": probs["no"]} if it["type"] == "noul"
                         else {n: probs[n] for n in it["options"]}))
uniform_td = [(it, {n: 1 / len(it["options"]) for n in it["options"]}) for it in td_items]

table = pd.DataFrame({
    "JEPA single pass (dev)": {f"{k} accuracy": v["accuracy"] for k, v in dev_result.items()}
                              | {f"{k} KL": v["kl"] for k, v in dev_result.items()},
    "JEPA single pass (Typed Decisions)": {f"{k} accuracy": v["accuracy"] for k, v in td_result.items()}
                                          | {f"{k} KL": v["kl"] for k, v in td_result.items()},
    "Signpost-D-4B (Typed Decisions)": {f"{k} accuracy": v["accuracy"] for k, v in report(ref_rows).items()}
                                       | {f"{k} KL": v["kl"] for k, v in report(ref_rows).items()},
    "Uniform (Typed Decisions)": {f"{k} accuracy": v["accuracy"] for k, v in report(uniform_td).items()}
                                 | {f"{k} KL": v["kl"] for k, v in report(uniform_td).items()},
})
pd.set_option("display.width", 250)
display(table)

print("RESULT | dev " + json.dumps(dev_result) + " | Typed Decisions " + json.dumps(td_result))
with open(OUT_DIR / "results.json", "w") as f:
    json.dump({"config": CONFIG, "best_epoch": state["best_epoch"], "dev": dev_result, "td": td_result,
               "reference": report(ref_rows), "history": state["history"]}, f, indent=1)
print("saved", OUT_DIR)